# Fusion diagnostics (no training)

Residual magnitudes, fusion-off and RadImageNet-mean ablations for the 5 CV fold models and the 20-epoch run,
on gold74 and 150 left-out studies per fold; parent Raptor on the same studies. Outputs in `diag/`.

Embedded source digests: `{"models/__init__.py": "360790aae4faf61c", "models/raptor_v5_radimagenet.py": "56ea1360da1a3958", "raptor_data/__init__.py": "238fa14a4e9c5976", "raptor_data/ready_tensors.py": "46dbbfdd78d2bf52", "raptor_data/cv_folds.py": "9862db030c789446", "train_fusion_v2.py": "f2d8a3f9ae78678f", "fusion_diagnostics.py": "139f45a281ef42b1"}`


In [ ]:
import importlib.metadata, subprocess, sys
try:
    installed = importlib.metadata.version('timm')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != '1.0.22':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm==1.0.22'])
assert importlib.metadata.version('timm') == '1.0.22'
import torch
print('torch', torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], flush=True)


In [ ]:
from pathlib import Path
SOURCES = {'models/__init__.py': '"""Trainable models for RSNA knee experiments."""\n', 'models/raptor_v5_radimagenet.py': '"""Raptor v5 with residual RadImageNet fusion inside CoAtNet.\n\nInput is an already preprocessed [B, K, 3, 384, 384] triplet bag. Both\nencoders receive exactly the same tensor; preprocessing belongs to the loader.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom contextlib import contextmanager, nullcontext\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\nfrom torch import Tensor, nn\nfrom torch.utils.checkpoint import checkpoint\n\n\nCOATNET_ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"\nLABELS = (\n    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s",\n    "Contusion", "Fracture",\n)\n\n\nRAD_INPUTS = ("shared", "pm1_triplet", "pm1_middle")\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\n\n@dataclass(frozen=True)\nclass FusionConfig:\n    stage2_hidden: int = 1024\n    stage4_hidden: int = 2048\n    mlp_dropout: float = 0.0\n    head_dropout: float = 0.2\n    gradient_checkpointing: bool = False\n    # What the RadImageNet trunk receives (CoAtNet always gets the ImageNet-normalized triplet):\n    # \'shared\' = the same triplet (original behaviour and the default for older checkpoints);\n    # \'pm1_triplet\' = the three slices rescaled to [-1, 1];\n    # \'pm1_middle\' = the middle slice in [-1, 1] copied to 3 channels (baseline Rad convention).\n    rad_input: str = "shared"\n\n\ndef rad_input_transform(image: Tensor, mode: str) -> Tensor:\n    """Map an ImageNet-normalized triplet [N,3,H,W] to the RadImageNet input convention."""\n    if mode == "shared":\n        return image\n    if mode not in RAD_INPUTS:\n        raise ValueError(f"Unknown rad_input {mode!r}; expected one of {RAD_INPUTS}")\n    mean = torch.tensor(IMAGENET_MEAN, device=image.device, dtype=image.dtype).view(1, 3, 1, 1)\n    std = torch.tensor(IMAGENET_STD, device=image.device, dtype=image.dtype).view(1, 3, 1, 1)\n    pm1 = (image*std + mean)*2 - 1          # exact inverse of the ImageNet normalization, then [-1, 1]\n    if mode == "pm1_middle":\n        pm1 = pm1[:, 1:2].expand(-1, 3, -1, -1)\n    return pm1.contiguous()\n\n\n@contextmanager\ndef preserve_batchnorm_buffers(module: nn.Module):\n    """Restore running statistics after a probe or checkpoint recomputation.\n\n    BatchNorm, SyncBatchNorm and timm BatchNormAct subclasses keep their normal\n    training behavior (including synchronization and trainable affine weights).\n    Only running_mean, running_var and num_batches_tracked are restored, so a\n    recomputed forward contributes gradients but does not count as a second\n    observation. Enter this context for recomputation only, after the original\n    forward has updated the statistics.\n    """\n    saved = [\n        (buffer, buffer.detach().clone())\n        for layer in module.modules()\n        if isinstance(layer, nn.modules.batchnorm._BatchNorm)\n        for name in ("running_mean", "running_var", "num_batches_tracked")\n        if (buffer := getattr(layer, name, None)) is not None\n    ]\n    try:\n        yield\n    finally:\n        with torch.no_grad():\n            for buffer, value in saved:\n                buffer.copy_(value)\n\n\nclass ConcatenationMLP(nn.Module):\n    """Three affine layers with GELU, applied to each spatial position.\n\n    Concatenation precedes every learned operation. Only the output projection\n    is zero-initialized, so the residual is exactly zero without a dead MLP.\n    """\n\n    def __init__(self, first_channels: int, second_channels: int,\n                 out_channels: int, hidden: int, dropout: float = 0.0):\n        super().__init__()\n        if min(first_channels, second_channels, out_channels, hidden) <= 0:\n            raise ValueError("MLP channel dimensions must be positive")\n        self.first_channels = first_channels\n        self.second_channels = second_channels\n        self.mlp = nn.Sequential(\n            nn.LayerNorm(first_channels + second_channels),\n            nn.Linear(first_channels + second_channels, hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, hidden),\n            nn.GELU(), nn.Dropout(dropout),\n            nn.Linear(hidden, out_channels),\n        )\n        nn.init.zeros_(self.mlp[-1].weight)\n        nn.init.zeros_(self.mlp[-1].bias)\n\n    def forward(self, first: Tensor, second: Tensor) -> Tensor:\n        if (first.ndim != 4 or second.ndim != 4\n                or first.shape[0] != second.shape[0]\n                or first.shape[2:] != second.shape[2:]\n                or first.shape[1] != self.first_channels\n                or second.shape[1] != self.second_channels):\n            raise ValueError(f"Incompatible fusion features: {first.shape}, {second.shape}")\n        joined = torch.cat((first, second), dim=1).permute(0, 2, 3, 1)\n        return self.mlp(joined).permute(0, 3, 1, 2).contiguous()\n\n\nclass RadImageNetEncoder(nn.Module):\n    """The same torchvision ResNet-50 trunk used by the baseline Rad encoder.\n\n    Keep `backbone.0` ... `backbone.7` names for strict ResNet50.pt loading.\n    conv3_x is backbone[5] (torchvision layer2); conv5_x is backbone[7]\n    (layer4). No ImageNet weights are substituted for RadImageNet weights.\n    """\n\n    def __init__(self):\n        super().__init__()\n        from torchvision.models import resnet50\n\n        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])\n\n    def forward(self, image: Tensor) -> tuple[Tensor, Tensor]:\n        x = image\n        conv3 = None\n        for index, layer in enumerate(self.backbone):\n            x = layer(x)\n            if index == 5:\n                conv3 = x\n        return conv3, x\n\n\nclass FusedCoAtNet(nn.Module):\n    """Stage2 Z += F1(X,Z), Stage4 V += F2(V,Y), then normal CoAtNet pooling."""\n\n    def __init__(self, coatnet: nn.Module, radimagenet: nn.Module,\n                 config: FusionConfig = FusionConfig()):\n        super().__init__()\n        self.coatnet = coatnet\n        self.radimagenet = radimagenet\n        self.num_features = coatnet.num_features\n        channels = [entry["num_chs"] for entry in coatnet.feature_info]\n        if len(coatnet.stages) != 4 or channels[-4:] != [128, 256, 512, 1024]:\n            raise ValueError("Expected the four-stage Raptor v5 CoAtNet RMLP-2")\n        self.f1 = ConcatenationMLP(512, 256, 256, config.stage2_hidden, config.mlp_dropout)\n        self.f2 = ConcatenationMLP(1024, 2048, 1024, config.stage4_hidden, config.mlp_dropout)\n        self.gradient_checkpointing = config.gradient_checkpointing\n        if config.rad_input not in RAD_INPUTS:\n            raise ValueError(f"Unknown rad_input {config.rad_input!r}; expected one of {RAD_INPUTS}")\n        self.rad_input = config.rad_input\n        self.checkpoint_parts: frozenset[str] = frozenset()\n        self.requires_grad_(True)\n        self.coatnet.set_grad_checkpointing(config.gradient_checkpointing)\n\n    CHECKPOINT_PARTS = ("rad", "s1", "s2", "s3", "s4", "fusion")\n\n    def set_checkpoint_parts(self, parts) -> None:\n        """Runtime-only activation recomputation for selected whole sections.\n\n        Parts: rad (ResNet-50), s1 (stem+S1), s2, s3, s4 (CoAtNet stages) and\n        fusion (both MLPs). Not stored in checkpoints; numerics are unchanged.\n        These whole-section checkpoints preserve BatchNorm running buffers\n        during recomputation. This does not cover timm\'s separate internal\n        checkpointing; leave config.gradient_checkpointing=False when using\n        trainable BatchNorm statistics with this path.\n        """\n        parts = frozenset(parts)\n        unknown = parts - set(self.CHECKPOINT_PARTS)\n        if unknown:\n            raise ValueError(f"Unknown checkpoint parts: {sorted(unknown)}")\n        self.checkpoint_parts = parts\n\n    def _run(self, part: str, fn, *args):\n        if part in self.checkpoint_parts and self.training and torch.is_grad_enabled():\n            # s1 is a bound method spanning stem and stage 1; the other sections\n            # are individual modules. Fusion MLPs have no BatchNorm buffers.\n            scope = fn if isinstance(fn, nn.Module) else self.coatnet\n            return checkpoint(\n                fn, *args, use_reentrant=False,\n                context_fn=lambda: (nullcontext(), preserve_batchnorm_buffers(scope)),\n            )\n        return fn(*args)\n\n    def _stem_s1(self, image: Tensor) -> Tensor:\n        return self.coatnet.stages[0](self.coatnet.stem(image))\n\n    def _fuse(self, mlp: nn.Module, first: Tensor, second: Tensor) -> Tensor:\n        if ((self.gradient_checkpointing or "fusion" in self.checkpoint_parts)\n                and self.training and torch.is_grad_enabled()):\n            return checkpoint(mlp, first, second, use_reentrant=False)\n        return mlp(first, second)\n\n    def forward_features(self, image: Tensor) -> Tensor:\n        if image.ndim != 4 or tuple(image.shape[1:]) != (3, 384, 384):\n            raise ValueError("Both backbones require [N, 3, 384, 384] preprocessed triplets")\n        # CoAtNet gets the triplet as saved; RadImageNet gets it in its configured convention.\n        x, y = self._run("rad", self.radimagenet, rad_input_transform(image, self.rad_input))\n        z = self._run("s1", self._stem_s1, image)\n        z = self._run("s2", self.coatnet.stages[1], z)  # S2 convolution, [N,256,48,48]\n        z = z + self._fuse(self.f1, x, z)  # F1 input order is (X,Z)\n        v = self._run("s3", self.coatnet.stages[2], z)  # S3 receives the UPDATED Z\n        v = self._run("s4", self.coatnet.stages[3], v)  # S4 transformer, [N,1024,12,12]\n        v = v + self._fuse(self.f2, v, y)  # F2 input order is (V,Y)\n        return self.coatnet.norm(v)\n\n    def forward(self, image: Tensor) -> Tensor:\n        return self.coatnet.forward_head(self.forward_features(image))\n\n\nclass RaptorClassifier(nn.Module):\n    """Original Raptor attention-MIL head, retaining checkpoint parameter names."""\n\n    def __init__(self, backbone: nn.Module, F_dim: int, n: int = 12, drop: float = 0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x: Tensor) -> Tensor:\n        if x.ndim != 5 or x.shape[0] == 0 or x.shape[1] == 0:\n            raise ValueError("Raptor expects a nonempty [B,K,3,384,384] triplet bag")\n        b, k = x.shape[:2]\n        return self.backbone(x.flatten(0, 1)).reshape(b, k, -1)\n\n    def head(self, feats: Tensor) -> Tensor:\n        h = self.norm(feats)\n        a = torch.softmax(self.att(h), dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        return (pooled * self.clsW).sum(-1) + self.clsb\n\n    def forward(self, x: Tensor) -> Tensor:\n        return self.head(self.encode(x))\n\n\ndef _checkpoint(source: str | Path | Mapping[str, Any]) -> Mapping[str, Any]:\n    return source if isinstance(source, Mapping) else torch.load(\n        source, map_location="cpu", weights_only=True,\n    )\n\n\ndef build_raptor_v5_radimagenet(\n    raptor_checkpoint: str | Path | Mapping[str, Any],\n    radimagenet_checkpoint: str | Path | Mapping[str, Any],\n    config: FusionConfig = FusionConfig(),\n) -> RaptorClassifier:\n    """Strict-load both pretrained parents, add zero residuals, unfreeze all.\n\n    Raptor checkpoint: {\'model\': state_dict, \'arch\': ..., \'res\': 384}.\n    Rad checkpoint: the baseline ResNet50.pt state dict, or {\'state_dict\': ...}.\n    Load and resume a *fused* checkpoint using load_fused_checkpoint instead.\n    """\n    import timm\n\n    ck = _checkpoint(raptor_checkpoint)\n    if ck.get("arch", COATNET_ARCH) != COATNET_ARCH or int(ck.get("res", 384)) != 384:\n        raise ValueError("Parent checkpoint must be Raptor v5 CoAtNet RMLP-2 at res=384")\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    model = RaptorClassifier(coatnet, F_dim=coatnet.num_features, drop=config.head_dropout)\n    model.load_state_dict(ck["model"], strict=True)\n    rad = RadImageNetEncoder()\n    rad_ck = _checkpoint(radimagenet_checkpoint)\n    rad.load_state_dict(rad_ck.get("state_dict", rad_ck), strict=True)\n    model.backbone = FusedCoAtNet(coatnet, rad, config)\n    model.fusion_config = config\n    model.requires_grad_(True)\n    return model\n\n\ndef optimizer_groups(model: RaptorClassifier, backbone_lr: float = 1e-5,\n                     fusion_lr: float = 1e-4, head_lr: float = 1e-4) -> list[dict[str, Any]]:\n    """All parameters, exactly once, with both backbones unfrozen."""\n    if not isinstance(model.backbone, FusedCoAtNet):\n        raise TypeError("Expected the fused Raptor model")\n    if any(not p.requires_grad for p in model.parameters()):\n        raise ValueError("Both backbones and the fusion/head must remain unfrozen")\n    groups = {"coatnet": [], "radimagenet": [], "fusion": [], "head": []}\n    for name, parameter in model.named_parameters():\n        if name.startswith("backbone.coatnet."):\n            group = "coatnet"\n        elif name.startswith("backbone.radimagenet."):\n            group = "radimagenet"\n        elif name.startswith(("backbone.f1.", "backbone.f2.")):\n            group = "fusion"\n        else:\n            group = "head"\n        groups[group].append(parameter)\n    rates = dict(coatnet=backbone_lr, radimagenet=backbone_lr, fusion=fusion_lr, head=head_lr)\n    return [{"name": name, "params": parameters, "lr": rates[name]}\n            for name, parameters in groups.items()]\n\n\ndef fused_checkpoint(model: RaptorClassifier) -> dict[str, Any]:\n    """Serialize model/config; trainers can add optimizer, scaler and epoch.\n\n    Tensors are copied to CPU: state_dict() alone shares storage with the live parameters, so\n    swapping weights back afterwards (e.g. EMA -> raw) would silently change the checkpoint.\n    """\n    state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}\n    return {"format": "raptor_v5_radimagenet_v1", "arch": COATNET_ARCH, "res": 384,\n            "labels": list(LABELS), "fusion_config": asdict(model.fusion_config),\n            "model": state}\n\n\ndef load_fused_checkpoint(source: str | Path | Mapping[str, Any]) -> RaptorClassifier:\n    import timm\n\n    ck = _checkpoint(source)\n    if (ck.get("format") != "raptor_v5_radimagenet_v1" or ck.get("arch") != COATNET_ARCH\n            or ck.get("res") != 384 or ck.get("labels") != list(LABELS)):\n        raise ValueError("Unsupported fused checkpoint architecture/input/label contract")\n    config = FusionConfig(**ck["fusion_config"])\n    coatnet = timm.create_model(COATNET_ARCH, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")\n    backbone = FusedCoAtNet(coatnet, RadImageNetEncoder(), config)\n    model = RaptorClassifier(backbone, F_dim=backbone.num_features, drop=config.head_dropout)\n    model.fusion_config = config\n    model.load_state_dict(ck["model"], strict=True)\n    model.requires_grad_(True)\n    return model\n', 'raptor_data/__init__.py': '"""Series-aware preprocessing for the fused Raptor model."""\n', 'raptor_data/ready_tensors.py': '"""Load exported tensors directly; performs no image preprocessing."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\n\ndef per_slot_indices(slot_of_window, k, generator=None):\n    """Choose k window indices, spreading them as evenly as possible over slots.\n\n    slot_of_window: 1-D integer tensor giving each candidate window\'s slot.\n    Slots are filled round-robin in a random order, one window at a time, up to\n    each slot\'s capacity, so six present slots with k=12 get two windows each.\n    Within a slot, windows are drawn without replacement. Only when the study\n    has fewer than k candidate windows are extra windows repeated.\n    """\n    slot_of_window = torch.as_tensor(slot_of_window).long()\n    if slot_of_window.ndim != 1 or not len(slot_of_window):\n        raise ValueError(\'Need a nonempty 1-D slot assignment\')\n    if k < 1:\n        raise ValueError(\'k must be positive\')\n    slots = torch.unique(slot_of_window).tolist()\n    members = {s: torch.nonzero(slot_of_window == s).flatten() for s in slots}\n    quota = {s: 0 for s in slots}\n    remaining = k\n    while remaining:\n        order = torch.randperm(len(slots), generator=generator).tolist()\n        progressed = False\n        for position in order:\n            s = slots[position]\n            if quota[s] < len(members[s]) and remaining:\n                quota[s] += 1\n                remaining -= 1\n                progressed = True\n        if not progressed:\n            break\n    chosen = []\n    for s in slots:\n        if quota[s]:\n            pool = members[s]\n            chosen.append(pool[torch.randperm(len(pool), generator=generator)[:quota[s]]])\n    chosen = torch.cat(chosen)\n    if remaining:\n        extra = torch.randint(len(slot_of_window), (remaining,), generator=generator)\n        chosen = torch.cat([chosen, extra])\n    return chosen[torch.randperm(len(chosen), generator=generator)]\n\n\nclass ReadyTripletDataset(Dataset):\n    """Use K=12 for fixed-size training bags, or k=None for full study bags.\n\nAttach the preprocessing notebook outputs or published private shard datasets.\nImages are already cropped, resized, intensity-scaled, and ImageNet-normalized.\nThe stored float16 tensor is suitable for training under torch.autocast.\n\nwindows=\'all\' keeps every exported window; windows=\'pure\' keeps only windows\nwhose three slices come from one series (same_series), falling back to all\nwindows for a study that has none. sampling=\'uniform\' draws k windows\nuniformly; sampling=\'per_slot\' spreads k windows evenly over present slots.\n"""\n    def __init__(self, roots, *, split=\'train\', k=12, labels=None,\n                 sampling=\'uniform\', windows=\'all\'):\n        self.k = k\n        if k is not None and (not isinstance(k, int) or k < 1):\n            raise ValueError(\'k must be positive or None\')\n        if sampling not in (\'uniform\', \'per_slot\'):\n            raise ValueError(f\'Unknown sampling: {sampling}\')\n        if windows not in (\'all\', \'pure\'):\n            raise ValueError(f\'Unknown window policy: {windows}\')\n        self.sampling, self.windows = sampling, windows\n        roots = [Path(r) for r in roots]\n        tables = [pd.read_csv(p, dtype={\'StudyInstanceUID\': str})\n                  for root in roots for p in root.rglob(\'splits.csv\')]\n        if not tables:\n            raise ValueError(\'No preprocessing splits.csv found\')\n        splits = pd.concat(tables, ignore_index=True)\n        if splits.StudyInstanceUID.duplicated().any():\n            raise ValueError(\'Attach each shard once; duplicate study IDs found\')\n        wanted = set(splits.loc[splits[\'split\']==split, \'StudyInstanceUID\'])\n        self.files = sorted(p for root in roots for p in root.rglob(\'*.npz\') if p.stem in wanted)\n        if {p.stem for p in self.files} != wanted:\n            raise ValueError(\'Attached shard metadata and tensor files disagree\')\n        self.labels = labels\n\n    def __len__(self):\n        return len(self.files)\n\n    def __getitem__(self, index):\n        file = self.files[index]\n        with np.load(file, allow_pickle=False) as saved:\n            slots = torch.from_numpy(saved[\'triplet_slot_ids\'])\n            same_series = torch.from_numpy(saved[\'same_series\']).bool()\n            present = torch.from_numpy(saved[\'slot_present\'])\n            x = torch.from_numpy(saved[\'images\'])\n        if x.dtype != torch.float16 or x.ndim != 4 or tuple(x.shape[1:]) != (3, 384, 384) or not len(x):\n            raise ValueError(f\'Invalid preprocessed image contract: {file}\')\n        if len(slots) != len(x) or len(same_series) != len(x):\n            raise ValueError(f\'Window metadata length mismatch: {file}\')\n        candidates = torch.arange(len(x))\n        if self.windows == \'pure\' and same_series.any():\n            candidates = candidates[same_series]\n        if self.k is not None:\n            if self.sampling == \'per_slot\':\n                # A window\'s slot is the slot of its middle slice.\n                local = per_slot_indices(slots[candidates, 1], self.k)\n            else:\n                local = torch.randperm(len(candidates))[:self.k]\n                if len(local) < self.k:\n                    local = torch.cat([local, torch.randint(len(candidates), (self.k-len(local),))])\n            indices = candidates[local]\n        else:\n            indices = candidates\n        x, slots, same_series = x[indices], slots[indices], same_series[indices]\n        item = {\'images\': x, \'triplet_slot_ids\': slots, \'same_series\': same_series,\n                \'slot_present\': present, \'study_uid\': file.stem}\n        if self.labels is not None:\n            item[\'targets\'] = torch.as_tensor(self.labels[file.stem], dtype=torch.float32)\n        return item\n', 'raptor_data/cv_folds.py': '"""Deterministic K-fold split of the 4,333-study training corpus and the per-epoch CV metrics.\n\nThe held-out gold74 studies are a separate preprocessing split and never enter these folds.\nNo patient identifiers are available, so folds are study-level, stratified over the 12\nfindings with iterative multilabel stratification (Sechidis et al., 2011).\n\nValidation accuracy (the selection score) = mean of the gold74 macro AUC and the left-out-fold\nmacro AUC, computed separately for raw and EMA weights.\n"""\nimport numpy as np\n\nMETRIC_COLUMNS = (\'fold\', \'epoch\', \'optimizer_step\', \'weights\', \'gold74_macro_auc\', \'fold_macro_auc\',\n                  \'validation_accuracy\', \'training_loss\', \'note\')\n\n\ndef stratified_folds(study_ids, labels, n_folds=5, seed=20261010):\n    """Map each study ID to a fold in [0, n_folds); independent of the input order."""\n    ids = [str(u) for u in study_ids]\n    y = np.asarray(labels)\n    if y.ndim != 2 or len(ids) != len(y):\n        raise ValueError(\'labels must be [studies, findings] and match study_ids\')\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Duplicate study IDs\')\n    if not np.isin(y, (0, 1)).all():\n        raise ValueError(\'Stratification needs binary labels\')\n    order = np.argsort(np.array(ids))\n    ids = [ids[i] for i in order]\n    y = y[order].astype(np.int64)\n    n = len(ids)\n    rng = np.random.default_rng(seed)\n    visit = rng.permutation(n)\n    fold_of = np.full(n, -1, dtype=np.int64)\n    remaining = np.ones(n, dtype=bool)\n    capacity = np.full(n_folds, n/n_folds)                       # studies still wanted per fold\n    wanted = np.tile(y.sum(0)/n_folds, (n_folds, 1)).astype(float)   # positives still wanted per fold/label\n\n    def pick(candidates):\n        candidates = candidates[capacity[candidates] == capacity[candidates].max()]\n        return int(candidates[0] if len(candidates) == 1 else candidates[rng.integers(len(candidates))])\n\n    while remaining.any():\n        counts = y[remaining].sum(0)\n        present = np.flatnonzero(counts > 0)\n        if len(present) == 0:                                    # studies with no positive finding\n            for i in visit[remaining[visit]]:\n                fold = pick(np.arange(n_folds))\n                fold_of[i] = fold; remaining[i] = False; capacity[fold] -= 1\n            break\n        label = present[np.argmin(counts[present])]              # rarest remaining finding first\n        for i in visit[remaining[visit] & (y[visit, label] == 1)]:\n            need = wanted[:, label]\n            fold = pick(np.flatnonzero(need == need.max()))\n            fold_of[i] = fold; remaining[i] = False\n            capacity[fold] -= 1; wanted[fold] -= y[i]\n    return dict(zip(ids, fold_of.tolist()))\n\n\ndef combined_score(by_split):\n    """Validation accuracy: mean of gold74 and left-out-fold macro AUC (gold74 alone if no fold)."""\n    gold = by_split[\'gold74\'][\'macro_auc\']\n    if \'validation\' not in by_split:\n        return gold\n    return (gold + by_split[\'validation\'][\'macro_auc\'])/2\n\n\ndef metric_rows(history, fold):\n    """One row per (evaluation, weights); step 0 is the parent, where EMA weights equal raw weights."""\n    rows = []\n    for entry in history:\n        evaluation = entry.get(\'evaluation\') or {}\n        kinds = dict(evaluation)\n        note = \'\'\n        if entry[\'epoch\'] == 0 and \'ema\' not in kinds and \'raw\' in kinds:\n            kinds[\'ema\'] = kinds[\'raw\']\n            note = \'step 0 (parent): EMA is initialised from these weights, so EMA = raw\'\n        for kind in (\'raw\', \'ema\'):\n            if kind not in kinds:\n                continue\n            by_split = kinds[kind]\n            rows.append(dict(\n                fold=fold, epoch=entry[\'epoch\'], optimizer_step=entry[\'step\'], weights=kind,\n                gold74_macro_auc=by_split[\'gold74\'][\'macro_auc\'],\n                fold_macro_auc=by_split[\'validation\'][\'macro_auc\'] if \'validation\' in by_split else None,\n                validation_accuracy=combined_score(by_split) if \'validation\' in by_split else None,\n                training_loss=entry.get(\'training_loss\'), note=note))\n    return rows\n', 'train_fusion_v2.py': '"""Fusion fine-tune v2: fp16, frozen BN statistics, per-slot K=12 bags, GPU augmentation,\nwarmup+cosine, EMA, and gold74/validation AUC evaluation including a step-0 parent control.\n\nImages are the saved, already-normalized float16 tensors. Augmentation is applied on the GPU\nto training bags only; evaluation sees the saved tensors unchanged.\n"""\nimport copy\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Subset\n\nfrom models.raptor_v5_radimagenet import (LABELS, FusionConfig, FusedCoAtNet, build_raptor_v5_radimagenet,\n                                          optimizer_groups, fused_checkpoint, load_fused_checkpoint)\nfrom raptor_data.ready_tensors import ReadyTripletDataset\n\nRECIPE_SHA = \'26ba1f6d2e41f37150239c8d46f738f2a0eca9e6b5a89a7542724c9f4c44aaca\'\nRAD_SHA = \'08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734\'\nDEFAULTS = dict(\n    epochs=4, k=12, sampling=\'per_slot\', train_windows=\'pure\', eval_windows=\'pure\',\n    seed=20261009, validation_fraction=0.1, accumulate=16, workers=3, gpus=1,\n    backbone_lr=1e-5, fusion_lr=1e-4, head_lr=3e-5, weight_decay=0.01,\n    warmup_steps=50, min_lr_ratio=0.01, ema_decay=0.998, clip_norm=3.0,\n    amp=\'fp16\', freeze_bn_stats=True, checkpoint_parts=\'auto\', attention=\'auto\', memory_limit_gib=13.5,\n    augment=True, aug_rotate_deg=10.0, aug_scale=0.10, aug_shift=0.05,\n    aug_contrast=0.10, aug_gamma=0.10, aug_brightness=0.05, aug_reverse_p=0.5,\n    eval_chunk=16, eval_at_start=True, validation_eval=\'final\',\n    max_hours=10.5, save_every_steps=100, resume_path=None,\n    smoke=False, smoke_updates=25, max_skipped_microbatches=20, rad_input=\'pm1_middle\',\n)\nCHECKPOINT_LADDER = ((), (\'s3\', \'s4\'), (\'s2\', \'s3\', \'s4\', \'fusion\'),\n                     (\'rad\', \'s1\', \'s2\', \'s3\', \'s4\', \'fusion\'))\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\n# ----------------------------------------------------------------------------- utilities\ndef event(kind, **data):\n    record = dict(event=kind, utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()), **data)\n    print(json.dumps(record), flush=True)\n    with Path(\'training_events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n        stream.write(json.dumps(record)+\'\\n\')\n    Path(\'progress.json\').write_text(json.dumps(record, indent=2), encoding=\'utf-8\')\n\n\ndef atomic_save(data, path):\n    path = Path(path)\n    temporary = path.with_suffix(\'.tmp\')\n    torch.save(data, temporary)\n    temporary.replace(path)\n\n\ndef freeze_bn_stats(model):\n    """Put every BatchNorm in eval mode: running statistics are used and never updated.\n    Affine weights stay trainable. Call after every model.train()."""\n    count = 0\n    for module in model.modules():\n        if isinstance(module, torch.nn.modules.batchnorm._BatchNorm):\n            module.eval()\n            count += 1\n    return count\n\n\ndef set_train_mode(model, cfg):\n    model.train()\n    return freeze_bn_stats(model) if cfg[\'freeze_bn_stats\'] else 0\n\n\ndef lr_lambda_factory(warmup, total, min_ratio):\n    warmup = max(0, int(warmup))\n    total = max(total, warmup+1)\n\n    def factor(step):\n        if warmup and step < warmup:\n            return min_ratio + (1-min_ratio)*(step+1)/warmup\n        progress = min(1.0, (step-warmup)/max(1, total-warmup))\n        return min_ratio + (1-min_ratio)*0.5*(1+math.cos(math.pi*progress))\n    return factor\n\n\nclass EMA:\n    """Float32 exponential moving average of all parameters (buffers are copied; BN is frozen)."""\n    def __init__(self, model, decay):\n        self.decay = decay\n        self.params = [p.detach().clone().float() for p in model.parameters()]\n\n    @torch.no_grad()\n    def update(self, model, step):\n        d = min(self.decay, (1+step)/(10+step))\n        current = [p.detach().float() if p.dtype != torch.float32 else p.detach() for p in model.parameters()]\n        torch._foreach_mul_(self.params, d)\n        torch._foreach_add_(self.params, current, alpha=1-d)\n\n    @torch.no_grad()\n    def swap_in(self, model):\n        backup = [p.detach().to(\'cpu\', copy=True) for p in model.parameters()]\n        for p, e in zip(model.parameters(), self.params):\n            p.copy_(e)\n        return backup\n\n    @torch.no_grad()\n    def restore(self, model, backup):\n        for p, b in zip(model.parameters(), backup):\n            p.copy_(b.to(p.device))\n\n    def state_dict(self):\n        return dict(decay=self.decay, params=[p.cpu() for p in self.params])\n\n    def load_state_dict(self, state, device):\n        self.decay = state[\'decay\']\n        self.params = [p.to(device) for p in state[\'params\']]\n\n\nclass AutocastModule(torch.nn.Module):\n    """Runs the wrapped module under autocast inside each DataParallel worker thread."""\n    def __init__(self, module, dtype, enabled=True):\n        super().__init__()\n        self.module, self.dtype, self.enabled = module, dtype, enabled\n\n    def forward(self, *args):\n        with torch.autocast(\'cuda\', dtype=self.dtype, enabled=self.enabled):\n            return self.module(*args)\n\n\n# ----------------------------------------------------------------------------- augmentation\ndef gpu_augment(bag, cfg, generator=None):\n    """bag: [B,K,3,H,W] normalized tensor on GPU. Returns augmented tensor, same dtype/shape.\n\n    Per window: random channel-order reversal (slice order), rotation/scale/shift, and\n    contrast/brightness/gamma in [0,1] intensity space. No horizontal flip (medial/lateral)."""\n    shape, dtype, device = bag.shape, bag.dtype, bag.device\n    x = bag.reshape(-1, *shape[-3:]).float()\n    n = x.shape[0]\n    mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=device).view(1, 3, 1, 1)\n    u = x*std + mean\n\n    def uniform(lo, hi):\n        return lo + (hi-lo)*torch.rand(n, device=device, generator=generator)\n\n    reverse = torch.rand(n, device=device, generator=generator) < cfg[\'aug_reverse_p\']\n    if reverse.any():\n        u = torch.where(reverse.view(n, 1, 1, 1), u.flip(1), u)\n    angle = uniform(-cfg[\'aug_rotate_deg\'], cfg[\'aug_rotate_deg\'])*math.pi/180\n    scale = uniform(1-cfg[\'aug_scale\'], 1+cfg[\'aug_scale\'])\n    tx = uniform(-cfg[\'aug_shift\'], cfg[\'aug_shift\'])*2   # grid coordinates span [-1, 1]\n    ty = uniform(-cfg[\'aug_shift\'], cfg[\'aug_shift\'])*2\n    cos, sin = torch.cos(angle)/scale, torch.sin(angle)/scale\n    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)\n    grid = F.affine_grid(theta, list(u.shape), align_corners=False)\n    u = F.grid_sample(u, grid, mode=\'bilinear\', padding_mode=\'zeros\', align_corners=False)\n    gamma = torch.exp(uniform(-math.log1p(cfg[\'aug_gamma\']), math.log1p(cfg[\'aug_gamma\']))).view(n, 1, 1, 1)\n    contrast = uniform(1-cfg[\'aug_contrast\'], 1+cfg[\'aug_contrast\']).view(n, 1, 1, 1)\n    brightness = uniform(-cfg[\'aug_brightness\'], cfg[\'aug_brightness\']).view(n, 1, 1, 1)\n    u = u.clamp(0, 1).pow(gamma)\n    u = ((u-0.5)*contrast + 0.5 + brightness).clamp(0, 1)\n    return ((u-mean)/std).to(dtype).reshape(shape)\n\n\n# ----------------------------------------------------------------------------- data/labels\ndef discover_roots():\n    roots = []\n    for home in [Path(\'/kaggle/input\'), Path(\'/kaggle/input/datasets/zafarnematov\')]:\n        roots.extend(home.glob(\'rsna-six-slot-raptor-v5-shard-*\'))\n    for home in [Path(\'/kaggle/input\'), Path(\'/kaggle/input/notebooks/zafarnematov\')]:\n        for notebook in home.glob(\'rsna-six-slot-raptor-v5-preprocessing-*\'):\n            roots.extend(notebook.glob(\'six_slot_raptor_shard_*\'))\n    roots = sorted(set(p for p in roots if (p/\'recipe.json\').is_file()))\n    if len(roots) != 32:\n        raise ValueError(f\'Expected all 32 completed shards; found {len(roots)}\')\n    for root in roots:\n        recipe = json.loads((root/\'recipe.json\').read_text())\n        manifest = json.loads((root/\'manifest.json\').read_text())\n        if recipe[\'sha256\'] != RECIPE_SHA or not recipe[\'normalized\'] or not manifest[\'complete\']:\n            raise ValueError(f\'Incorrect/incomplete preprocessing shard: {root}\')\n    return roots\n\n\ndef asset(slug, filename, base=\'/kaggle/input\'):\n    paths = sorted(Path(base).rglob(filename))\n    paths = [p for p in paths if slug in str(p)]\n    if len(paths) != 1:\n        raise FileNotFoundError(f\'Expected one {slug}/{filename}; found {paths}\')\n    return paths[0]\n\n\ndef load_label_table(path, required_ids, allow_missing=False, name=\'labels\'):\n    table = pd.read_csv(path, dtype={\'StudyInstanceUID\': str})\n    if table.StudyInstanceUID.duplicated().any():\n        raise ValueError(f\'Duplicate study IDs in {name}\')\n    table = table.set_index(\'StudyInstanceUID\')\n    missing = set(required_ids) - set(table.index)\n    if missing:\n        raise ValueError(f\'{name} lacks {len(missing)} required studies\')\n    values = table.loc[list(required_ids), list(LABELS)].to_numpy(dtype=np.float32)\n    finite = np.isfinite(values)\n    if not allow_missing and not finite.all():\n        raise ValueError(f\'Missing values in {name}\')\n    if not ((values[finite] >= 0) & (values[finite] <= 1)).all():\n        raise ValueError(f\'Out-of-range values in {name}\')\n    return dict(zip(required_ids, values))\n\n\ndef split_indices(n, seed, fraction):\n    order = np.random.default_rng(seed).permutation(n)\n    nval = round(n*fraction)\n    if not 0 < nval < n:\n        raise ValueError(\'Validation fraction must leave nonempty training and validation sets\')\n    return sorted(order[nval:].tolist()), sorted(order[:nval].tolist())\n\n\ndef macro_auc(y_true, y_score):\n    """Per-finding ROC AUC on available (non-NaN) labels; findings with one class are skipped."""\n    from sklearn.metrics import roc_auc_score\n    per = {}\n    for j, name in enumerate(LABELS):\n        mask = np.isfinite(y_true[:, j])\n        truth = y_true[mask, j]\n        if mask.sum() and 0 < truth.sum() < len(truth):\n            per[name] = float(roc_auc_score(truth, y_score[mask, j]))\n    return (float(np.mean(list(per.values()))) if per else float(\'nan\')), per\n\n\n# ----------------------------------------------------------------------------- evaluation\n@torch.inference_mode()\ndef predict(model, dataset, cfg, dtype, workers):\n    """Full-bag logits for every study in dataset, splitting each bag over all GPUs."""\n    model.eval()\n    devices = list(range(min(torch.cuda.device_count(), cfg[\'gpus\'])))\n    replicas = [model.backbone] + [copy.deepcopy(model.backbone).to(f\'cuda:{d}\') for d in devices[1:]]\n    wrapped = [AutocastModule(r, dtype) for r in replicas]\n    event(\'predict_started\', studies=len(dataset), devices=len(devices), chunk=cfg[\'eval_chunk\'], workers=workers)\n    loader = DataLoader(dataset, batch_size=1, num_workers=workers, pin_memory=True,\n                        prefetch_factor=2 if workers else None)\n    uids, logits = [], []\n    size = cfg[\'eval_chunk\']*len(devices)\n    started, wait_s, gpu_s = time.monotonic(), 0.0, 0.0\n    fetch = time.monotonic()\n    for index, batch in enumerate(loader):\n        got = time.monotonic()\n        wait_s += got-fetch\n        images = batch[\'images\'][0]\n        # Every chunk is padded to the same [eval_chunk*GPUs] shape (repeating the last window, whose\n        # features are then discarded) so cudnn autotuning runs once instead of per bag length.\n        n_windows = len(images)\n        padded = -(-n_windows//size)*size\n        if padded > n_windows:\n            images = torch.cat([images, images[-1:].expand(padded-n_windows, *images.shape[1:])])\n        features = []\n        for chunk in images.split(size):\n            inputs = [(p.to(f\'cuda:{d}\', non_blocking=True),) for p, d in zip(chunk.chunk(len(devices)), devices)]\n            if not all(torch.isfinite(x).all() for (x,) in inputs):\n                raise ValueError(\'Non-finite saved evaluation tensor\')\n            if len(devices) == 1:   # stay on the main thread: per-call threads reset cuDNN plan caches\n                outputs = [wrapped[0](inputs[0][0])]\n            else:\n                outputs = torch.nn.parallel.parallel_apply(wrapped, inputs, devices=devices)\n            features.extend(o.to(\'cuda:0\').float() for o in outputs)\n        with torch.autocast(\'cuda\', dtype=dtype):\n            out = model.head(torch.cat(features, 0)[:n_windows][None])\n        logits.append(out.float().cpu()[0])\n        uids.append(batch[\'study_uid\'][0])\n        gpu_s += time.monotonic()-got\n        if index in (0, 1, 2) or (index+1) % 25 == 0:\n            event(\'predict_progress\', studies=index+1, total=len(loader), windows=n_windows,\n                  seconds=round(time.monotonic()-started, 1), data_wait_s=round(wait_s, 1), gpu_s=round(gpu_s, 1))\n        fetch = time.monotonic()\n    del replicas, wrapped\n    torch.cuda.empty_cache()\n    return uids, torch.stack(logits).numpy()\n\n\ndef run_evaluation(tag, model, datasets, truth, cfg, dtype, step):\n    """datasets: name -> dataset; truth: name -> dict(binary=..., soft=...)."""\n    results = {}\n    Path(\'predictions\').mkdir(exist_ok=True)\n    for name, dataset in datasets.items():\n        started = time.monotonic()\n        uids, logits = predict(model, dataset, cfg, dtype, cfg[\'workers\'])\n        probs = 1/(1+np.exp(-logits))\n        frame = pd.DataFrame(probs, columns=list(LABELS))\n        frame.insert(0, \'StudyInstanceUID\', uids)\n        frame.to_csv(f\'predictions/{tag}_{name}.csv\', index=False)\n        y = np.stack([truth[name][\'binary\'][u] for u in uids])\n        auc, per = macro_auc(y, probs)\n        record = dict(macro_auc=auc, per_finding=per, studies=len(uids),\n                      seconds=round(time.monotonic()-started, 1))\n        if truth[name].get(\'soft\') is not None:\n            soft = np.stack([truth[name][\'soft\'][u] for u in uids])\n            record[\'soft_bce\'] = float(F.binary_cross_entropy_with_logits(\n                torch.from_numpy(logits), torch.from_numpy(soft)).item())\n        results[name] = record\n        event(\'evaluation\', tag=tag, split=name, step=step, **record)\n    return results\n\n\n# ----------------------------------------------------------------------------- memory/speed probe\ndef set_fused_attention(model, fused):\n    """Toggle timm\'s fused SDPA attention (True) vs explicit matmul attention (False).\n    Both compute the same function; speed differs by GPU/precision. Returns modules changed."""\n    count = 0\n    for module in model.modules():\n        if hasattr(module, \'fused_attn\'):\n            module.fused_attn = bool(fused)\n            count += 1\n    return count\n\n\ndef time_training_step(model, network, images, y, cfg, scaler, passes=3):\n    """Peak memory and per-pass seconds of forward+backward on one batch (no optimizer step)."""\n    for d in range(torch.cuda.device_count()):\n        torch.cuda.reset_peak_memory_stats(d)\n    timings = []\n    x = logits = loss = None\n    try:\n        for _ in range(passes):   # first pass includes cudnn autotuning\n            torch.cuda.synchronize()\n            started = time.monotonic()\n            x = gpu_augment(images, cfg) if cfg[\'augment\'] else images\n            logits = network(x)\n            loss = F.binary_cross_entropy_with_logits(logits.float(), y)\n            scaler.scale(loss).backward()\n            torch.cuda.synchronize()\n            timings.append(round(time.monotonic()-started, 2))\n            x = logits = loss = None\n            model.zero_grad(set_to_none=True)\n    finally:\n        x = logits = loss = None\n        model.zero_grad(set_to_none=True)\n        torch.cuda.empty_cache()\n    peak = [round(torch.cuda.max_memory_allocated(d)/2**30, 2) for d in range(torch.cuda.device_count())]\n    return peak, timings\n\n\ndef probe_checkpointing(model, network, batch, cfg, scaler, dtype):\n    """For each attention implementation, find the cheapest checkpoint setting that fits in memory;\n    then keep the (attention, checkpoint) pair with the fastest steady-state training step."""\n    ladder = CHECKPOINT_LADDER if cfg[\'checkpoint_parts\'] == \'auto\' else (tuple(cfg[\'checkpoint_parts\']),)\n    modes = {\'auto\': (True, False), \'fused\': (True,), \'math\': (False,)}[cfg[\'attention\']]\n    images = batch[\'images\'].cuda(non_blocking=True)\n    y = batch[\'targets\'].cuda(non_blocking=True)\n    report, candidates = [], []\n    for fused in modes:\n        set_fused_attention(model, fused)\n        for parts in ladder:\n            model.backbone.set_checkpoint_parts(parts)\n            entry = dict(fused_attention=fused, parts=list(parts))\n            try:\n                peak, timings = time_training_step(model, network, images, y, cfg, scaler)\n            except RuntimeError as error:\n                if \'out of memory\' not in str(error).lower():\n                    raise\n                torch.cuda.empty_cache()\n                report.append({**entry, \'oom\': True})\n                continue\n            fits = max(peak) <= cfg[\'memory_limit_gib\']\n            report.append({**entry, \'peak_gib\': peak, \'seconds\': timings, \'fits\': fits})\n            event(\'probe_result\', **report[-1])\n            if fits:\n                candidates.append((min(timings[1:]), fused, parts))\n                break\n    if not candidates:\n        event(\'checkpoint_probe\', results=report, chosen=None)\n        raise RuntimeError(\'No attention/activation-checkpoint setting fits in GPU memory\')\n    seconds, fused, parts = min(candidates, key=lambda c: c[0])\n    set_fused_attention(model, fused)\n    model.backbone.set_checkpoint_parts(parts)\n    event(\'checkpoint_probe\', results=report, chosen=dict(fused_attention=fused, parts=list(parts),\n                                                          seconds_per_microbatch=seconds))\n    return dict(fused_attention=fused, parts=list(parts))\n\n\n# ----------------------------------------------------------------------------- main\ndef main(config=None):\n    started = time.monotonic()\n    cfg = {**DEFAULTS, **(config or {})}\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU training requested; refusing CPU fallback\')\n    if cfg[\'amp\'] not in (\'fp16\', \'bf16\'):\n        raise ValueError(\'amp must be fp16 or bf16\')\n    random.seed(cfg[\'seed\']); np.random.seed(cfg[\'seed\']); torch.manual_seed(cfg[\'seed\'])\n    torch.cuda.manual_seed_all(cfg[\'seed\'])\n    torch.backends.cudnn.benchmark = True\n    dtype = torch.float16 if cfg[\'amp\'] == \'fp16\' else torch.bfloat16\n    capability = torch.cuda.get_device_capability(0)\n    if dtype == torch.bfloat16 and capability < (8, 0):\n        raise ValueError(f\'bf16 has no tensor-core support on compute capability {capability}; use fp16\')\n\n    roots = discover_roots()\n    all_train = ReadyTripletDataset(roots, split=\'train\', k=None)\n    gold_files = ReadyTripletDataset(roots, split=\'held_out_gold\', k=None).files\n    if len(all_train) != 4333 or len(gold_files) != 74:\n        raise ValueError(f\'Corpus split mismatch: {len(all_train)}, {len(gold_files)}\')\n    train_ids_all = [p.stem for p in all_train.files]\n    gold_ids = [p.stem for p in gold_files]\n    soft = load_label_table(asset(\'jev-knee-labels\', \'HYBRID_labels_scores.csv\'), train_ids_all, name=\'HYBRID soft\')\n    binary = load_label_table(asset(\'jev-knee-labels\', \'HYBRID_labels_binary.csv\'), train_ids_all, name=\'HYBRID binary\')\n    gold = load_label_table(asset(\'jev-knee-labels\', \'gold74_eval_set.csv\'), gold_ids, allow_missing=True, name=\'gold74\')\n\n    train_data = ReadyTripletDataset(roots, split=\'train\', k=cfg[\'k\'], labels=soft,\n                                     sampling=cfg[\'sampling\'], windows=cfg[\'train_windows\'])\n    val_full = ReadyTripletDataset(roots, split=\'train\', k=None, windows=cfg[\'eval_windows\'])\n    gold_data = ReadyTripletDataset(roots, split=\'held_out_gold\', k=None, windows=cfg[\'eval_windows\'])\n    gold_all_windows = ReadyTripletDataset(roots, split=\'held_out_gold\', k=None, windows=\'all\')\n    assert train_data.files == all_train.files == val_full.files\n    train_idx, val_idx = split_indices(len(train_data), cfg[\'seed\'], cfg[\'validation_fraction\'])\n    split = dict(train=[train_ids_all[i] for i in train_idx], validation=[train_ids_all[i] for i in val_idx],\n                 heldout_gold=gold_ids)\n    assert not (set(split[\'train\']) | set(split[\'validation\'])) & set(gold_ids)\n    Path(\'training_split.json\').write_text(json.dumps(split, indent=2))\n    Path(\'training_config.json\').write_text(json.dumps({**cfg, \'recipe_sha256\': RECIPE_SHA, \'labels\': list(LABELS),\n        \'target_source\': \'HYBRID_labels_scores.csv\', \'validation_binary\': \'HYBRID_labels_binary.csv\',\n        \'gold\': \'gold74_eval_set.csv (missing cells masked per finding)\'}, indent=2))\n    if cfg[\'validation_eval\'] not in (\'every\', \'final\', \'never\'):\n        raise ValueError(\'validation_eval must be every, final or never\')\n    gold_sets = {\'gold74\': gold_data}\n    validation_sets = {\'validation\': Subset(val_full, val_idx)}\n    truth = {\'gold74\': dict(binary=gold), \'gold74_allwindows\': dict(binary=gold),\n             \'validation\': dict(binary=binary, soft=soft)}\n    event(\'inputs_verified\', shards=len(roots), training=len(train_idx), validation=len(val_idx),\n          heldout_gold=len(gold_ids), gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n          capability=list(capability), config=cfg)\n\n    resume = None\n    if cfg[\'resume_path\']:\n        resume = torch.load(cfg[\'resume_path\'], map_location=\'cpu\', weights_only=True)\n        if resume[\'training_split\'] != split or resume[\'training_config\'][\'recipe_sha256\'] != RECIPE_SHA:\n            raise ValueError(\'Resume dataset/split differs\')\n        for key in (\'epochs\', \'k\', \'sampling\', \'seed\', \'accumulate\', \'backbone_lr\', \'fusion_lr\', \'head_lr\'):\n            if resume[\'training_config\'][key] != cfg[key]:\n                raise ValueError(f\'Resume setting differs: {key}\')\n        if resume[\'training_config\'].get(\'rad_input\', \'shared\') != cfg[\'rad_input\']:\n            raise ValueError(\'Resume setting differs: rad_input (set it to the checkpoint value)\')\n        model = load_fused_checkpoint(resume)\n        prior_best = Path(cfg[\'resume_path\']).with_name(\'fusion_best.pt\')\n        if prior_best.is_file() and not Path(\'fusion_best.pt\').exists():\n            import shutil\n            shutil.copyfile(prior_best, \'fusion_best.pt\')   # keep the previous session\'s best in this output\n            event(\'prior_best_carried\', source=str(prior_best))\n    else:\n        import hashlib\n        rad = asset(\'resnet-50-radimagenet-marwan\', \'ResNet50.pt\')\n        with rad.open(\'rb\') as stream:\n            if hashlib.file_digest(stream, \'sha256\').hexdigest() != RAD_SHA:\n                raise ValueError(\'RadImageNet asset hash differs from branch manifest\')\n        model = build_raptor_v5_radimagenet(asset(\'raptor-knee-maxspan\', \'raptor_ft_coatnet_v5_full_swa.pt\'),\n                                            rad, FusionConfig(gradient_checkpointing=False,\n                                                              rad_input=cfg[\'rad_input\']))\n    model = model.cuda()\n    bn_frozen = set_train_mode(model, cfg)\n    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg[\'backbone_lr\'], cfg[\'fusion_lr\'], cfg[\'head_lr\']),\n                                  weight_decay=cfg[\'weight_decay\'])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=dtype == torch.float16, init_scale=1024)\n    n_gpu = torch.cuda.device_count()\n    # One GPU by default: DataParallel/parallel_apply start new threads on every call, which on\n    # Kaggle T4s made each step ~8x slower than the single-thread benchmark (0.71 s per 12-window\n    # forward+backward). Effective batch = batch_size * accumulate.\n    batch_size = max(1, min(n_gpu, cfg[\'gpus\'], 2))\n    wrapped = AutocastModule(model, dtype)\n    network = torch.nn.DataParallel(wrapped, device_ids=list(range(batch_size))) if batch_size > 1 else wrapped\n    updates_per_epoch = math.ceil(math.ceil(len(train_idx)/batch_size)/cfg[\'accumulate\'])\n    total_updates = cfg[\'smoke_updates\'] if cfg[\'smoke\'] else cfg[\'epochs\']*updates_per_epoch\n    scheduler = torch.optim.lr_scheduler.LambdaLR(\n        optimizer, lr_lambda_factory(cfg[\'warmup_steps\'], total_updates, cfg[\'min_lr_ratio\']))\n    ema = EMA(model, cfg[\'ema_decay\'])\n\n    epoch_start, batch_start, global_step = 0, 0, 0\n    best = dict(gold_auc=-1.0)\n    history = []\n    carried = (0.0, 0)\n    if resume:\n        optimizer.load_state_dict(resume[\'optimizer\']); scaler.load_state_dict(resume[\'scaler\'])\n        scheduler.load_state_dict(resume[\'scheduler\']); ema.load_state_dict(resume[\'ema\'], \'cuda:0\')\n        epoch_start, batch_start = resume[\'epoch_index\'], resume[\'next_batch\']\n        global_step, best, history = resume[\'global_step\'], resume[\'best\'], resume[\'history\']\n        carried = (resume[\'epoch_running\'], resume[\'epoch_seen\'])\n        torch.set_rng_state(resume[\'torch_rng\']); torch.cuda.set_rng_state_all(resume[\'cuda_rng\'])\n        random.setstate(resume[\'python_rng\'])\n        del resume\n\n    # Memory probe on one real training batch (no optimizer step).\n    probe_loader = DataLoader(Subset(train_data, train_idx[:batch_size]), batch_size=batch_size, num_workers=0)\n    speed = probe_checkpointing(model, network, next(iter(probe_loader)), cfg, scaler, dtype)\n    event(\'model_ready\', trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),\n          frozen_batchnorm_layers=bn_frozen, **speed,\n          optimizer_groups=[dict(name=g[\'name\'], lr=g[\'lr\']) for g in optimizer.param_groups],\n          batch_size=batch_size, k=cfg[\'k\'], effective_batch=batch_size*cfg[\'accumulate\'],\n          updates_per_epoch=updates_per_epoch, total_updates=total_updates, amp=str(dtype))\n\n    def evaluate_all(tag, step, include_ema=True, validation=False):\n        eval_sets = {**gold_sets, **(validation_sets if validation and not cfg[\'smoke\'] else {})}\n        results = {\'raw\': run_evaluation(f\'{tag}_raw\', model, eval_sets, truth, cfg, dtype, step)}\n        if include_ema:\n            backup = ema.swap_in(model)\n            try:\n                results[\'ema\'] = run_evaluation(f\'{tag}_ema\', model, eval_sets, truth, cfg, dtype, step)\n            finally:\n                ema.restore(model, backup)\n        set_train_mode(model, cfg)\n        return results\n\n    def consider_best(results, tag, step):\n        nonlocal best\n        for kind, by_split in results.items():\n            auc = by_split[\'gold74\'][\'macro_auc\']\n            if auc > best[\'gold_auc\']:\n                best = dict(gold_auc=auc, weights=kind, tag=tag, step=step)\n                if kind == \'ema\':\n                    backup = ema.swap_in(model)\n                    try:\n                        ck = fused_checkpoint(model)   # copies taken while the EMA weights are installed\n                    finally:\n                        ema.restore(model, backup)\n                    source = ema.params\n                else:\n                    ck = fused_checkpoint(model)\n                    source = list(model.parameters())\n                ck.update(selection=best, training_config=cfg, recipe_sha256=RECIPE_SHA)\n                atomic_save(ck, \'fusion_best.pt\')\n                saved = torch.load(\'fusion_best.pt\', map_location=\'cpu\', weights_only=True)[\'model\']\n                names = [n for n, _ in model.named_parameters()]\n                if len(names) != len(source) or not all(\n                        torch.equal(saved[n].float(), s.detach().float().cpu()) for n, s in zip(names, source)):\n                    raise RuntimeError(\'fusion_best.pt does not contain the selected weights\')\n                del saved\n                event(\'best_updated\', **best)\n\n    def save(epoch, next_batch, running, seen):\n        ck = fused_checkpoint(model)\n        ck.update(optimizer=optimizer.state_dict(), scaler=scaler.state_dict(), scheduler=scheduler.state_dict(),\n                  ema=ema.state_dict(), epoch_index=epoch, next_batch=next_batch, global_step=global_step,\n                  best=best, history=history, epoch_running=running, epoch_seen=seen,\n                  training_split=split, training_config={**cfg, \'recipe_sha256\': RECIPE_SHA},\n                  torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all(),\n                  python_rng=random.getstate())\n        atomic_save(ck, \'fusion_last.pt\')\n        event(\'checkpoint_saved\', epoch_index=epoch, next_batch=next_batch, optimizer_steps=global_step)\n\n    if cfg[\'eval_at_start\'] and global_step == 0:\n        # Step 0: zero residuals => exactly the parent Raptor (EMA equals raw here).\n        start = evaluate_all(\'step0\', 0, include_ema=False, validation=cfg[\'validation_eval\'] == \'every\')\n        start[\'raw\'].update(run_evaluation(\'step0_raw\', model, {\'gold74_allwindows\': gold_all_windows},\n                                           truth, cfg, dtype, 0))\n        set_train_mode(model, cfg)\n        history.append(dict(step=0, epoch=0, evaluation=start))\n        consider_best({\'raw\': start[\'raw\']}, \'step0\', 0)\n\n    finished = False\n    for epoch in range(epoch_start, cfg[\'epochs\']):\n        generator = torch.Generator().manual_seed(cfg[\'seed\']+epoch)\n        order = torch.randperm(len(train_idx), generator=generator).tolist()\n        loader = DataLoader(Subset(train_data, [train_idx[i] for i in order]), batch_size=batch_size,\n                            num_workers=cfg[\'workers\'], pin_memory=True, drop_last=False,\n                            prefetch_factor=2 if cfg[\'workers\'] else None, persistent_workers=False)\n        optimizer.zero_grad(set_to_none=True)\n        running, seen = carried if epoch == epoch_start else (0.0, 0)\n        skipped, amp_skips = 0, 0\n        wait_s, compute_s, timed = 0.0, 0.0, 0\n        event(\'epoch_started\', epoch=epoch+1, batches=len(loader), resume_batch=batch_start)\n        fetch_start = time.monotonic()\n        for batch_index, batch in enumerate(loader):\n            if batch_index < batch_start:\n                fetch_start = time.monotonic()\n                continue\n            got = time.monotonic()\n            images = batch[\'images\'].cuda(non_blocking=True)\n            y = batch[\'targets\'].cuda(non_blocking=True)\n            if not torch.isfinite(images).all():\n                raise ValueError(\'Non-finite saved training tensor\')\n            if cfg[\'augment\']:\n                images = gpu_augment(images, cfg)\n            group_start = (batch_index//cfg[\'accumulate\'])*cfg[\'accumulate\']\n            group_size = min(cfg[\'accumulate\'], len(loader)-group_start)\n            logits = network(images)\n            loss = F.binary_cross_entropy_with_logits(logits.float(), y)\n            if torch.isfinite(loss):\n                scaler.scale(loss/group_size).backward()\n                running += float(loss.detach())*len(y); seen += len(y)\n            else:\n                skipped += 1\n                event(\'microbatch_skipped_nonfinite\', epoch=epoch+1, batch=batch_index+1, skipped=skipped)\n                if skipped > cfg[\'max_skipped_microbatches\']:\n                    raise FloatingPointError(\'Too many non-finite training losses\')\n            del images, logits, loss\n            torch.cuda.synchronize()\n            done = time.monotonic()\n            if batch_index > batch_start + 2:   # ignore warm-up iterations in timing\n                wait_s += got-fetch_start; compute_s += done-got; timed += 1\n            if batch_index < batch_start + 3:\n                event(\'microbatch_complete\', epoch=epoch+1, batch=batch_index+1, loss=running/max(seen, 1),\n                      seconds=round(done-fetch_start, 2),\n                      peak_gpu_gib=[round(torch.cuda.max_memory_allocated(i)/2**30, 2) for i in range(n_gpu)])\n            boundary = (batch_index+1) % cfg[\'accumulate\'] == 0 or batch_index+1 == len(loader)\n            if boundary:\n                scaler.unscale_(optimizer)\n                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg[\'clip_norm\'])\n                before = scaler.get_scale()\n                scaler.step(optimizer); scaler.update()\n                advanced = (not scaler.is_enabled()) or scaler.get_scale() >= before\n                optimizer.zero_grad(set_to_none=True)\n                if advanced:\n                    scheduler.step(); global_step += 1\n                    ema.update(model, global_step)\n                    amp_skips = 0\n                else:\n                    amp_skips += 1\n                    event(\'amp_update_skipped\', consecutive=amp_skips, scale=scaler.get_scale())\n                    if amp_skips >= 8:\n                        raise FloatingPointError(\'Eight consecutive non-finite AMP updates\')\n                if advanced and (global_step <= 3 or global_step % 10 == 0):\n                    per_mb = (wait_s+compute_s)/max(timed, 1)\n                    remaining_mb = (len(loader)-batch_index-1) + (cfg[\'epochs\']-epoch-1)*len(loader)\n                    if cfg[\'smoke\']:\n                        remaining_mb = max(0, (total_updates-global_step)*cfg[\'accumulate\'])\n                    event(\'training_progress\', epoch=epoch+1, batch=batch_index+1, batches=len(loader),\n                          optimizer_steps=global_step, loss=running/max(seen, 1), gradient_norm=float(grad_norm),\n                          lr=[g[\'lr\'] for g in optimizer.param_groups], scale=scaler.get_scale(),\n                          sec_per_microbatch=round(per_mb, 3),\n                          data_wait_fraction=round(wait_s/max(wait_s+compute_s, 1e-9), 3),\n                          projected_epoch_hours=round(per_mb*len(loader)/3600, 2),\n                          projected_remaining_training_hours=round(per_mb*remaining_mb/3600, 2),\n                          hours=round((time.monotonic()-started)/3600, 3),\n                          peak_gpu_gib=[round(torch.cuda.max_memory_allocated(i)/2**30, 2) for i in range(n_gpu)])\n                if advanced and global_step % cfg[\'save_every_steps\'] == 0:\n                    save(epoch, batch_index+1, running, seen)\n                if cfg[\'smoke\'] and global_step >= total_updates:\n                    finished = True\n                    break\n                if time.monotonic()-started > cfg[\'max_hours\']*3600:\n                    save(epoch, batch_index+1, running, seen)\n                    event(\'paused_time_budget\', epoch=epoch+1, next_batch=batch_index+1,\n                          optimizer_steps=global_step, resume_checkpoint=\'fusion_last.pt\')\n                    return\n            fetch_start = time.monotonic()\n        batch_start = 0\n        tag = \'smoke_end\' if finished else f\'epoch{epoch+1}\'\n        last_epoch = epoch+1 == cfg[\'epochs\']\n        results = evaluate_all(tag, global_step, validation=cfg[\'validation_eval\'] == \'every\'\n                               or (cfg[\'validation_eval\'] == \'final\' and last_epoch))\n        history.append(dict(step=global_step, epoch=epoch+1, training_loss=running/max(seen, 1),\n                            evaluation=results))\n        consider_best(results, tag, global_step)\n        if not cfg[\'smoke\']:\n            backup = ema.swap_in(model)\n            atomic_save(fused_checkpoint(model), f\'fusion_ema_epoch{epoch+1}.pt\')\n            ema.restore(model, backup)\n        Path(\'history.json\').write_text(json.dumps(history, indent=2))\n        save(epoch+1, 0, 0.0, 0)\n        summary = {kind: {name: r[\'macro_auc\'] for name, r in by.items()} for kind, by in results.items()}\n        event(\'epoch_complete\', epoch=epoch+1, optimizer_steps=global_step, training_loss=running/max(seen, 1),\n              macro_auc=summary, best=best)\n        if finished:\n            break\n        if time.monotonic()-started > cfg[\'max_hours\']*3600:\n            event(\'paused_time_budget\', epoch=epoch+1, optimizer_steps=global_step, resume_checkpoint=\'fusion_last.pt\')\n            return\n    event(\'training_complete\', optimizer_steps=global_step, best=best,\n          hours=round((time.monotonic()-started)/3600, 3))\n\n\nif __name__ == \'__main__\':\n    main(globals().get(\'RUN_CONFIG\'))\n', 'fusion_diagnostics.py': '"""Fusion diagnostics on Kaggle (2 GPUs, one process per GPU via DIAG_RANK / CUDA_VISIBLE_DEVICES).\n\nFor every trained checkpoint (5 CV folds, the 20-epoch run\'s saved best and final EMA):\n  * weights: drift from the parent per component; fusion layer norms (last layers were zero at init).\n  * activations, per window: ||F1 residual|| / ||S2 features||, cos(F1 residual, S2 features), the same\n    for F2 at stage 4, and the relative change of the pooled 1024-d feature when the fusion is removed.\n  * predictions in three modes on the same studies:\n        full       - the model as trained;\n        no_fusion  - both residuals removed (CoAtNet path of the fine-tuned model only);\n        rad_mean   - RadImageNet features replaced by their average over the evaluated windows, so the\n                     fusion MLPs still run but receive no study-specific RadImageNet information.\nEvaluation sets: gold74 (all checkpoints) and, for each CV fold model, 150 studies of its own left-out fold.\nThe parent Raptor (step 0) is evaluated on the same sets for a fine-tuning vs fusion decomposition.\nWrites /kaggle/working/diag/*. No training, no labels are used here (scoring happens offline).\n"""\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom models.raptor_v5_radimagenet import (LABELS, FusionConfig, build_raptor_v5_radimagenet,\n                                          load_fused_checkpoint, rad_input_transform)\nfrom raptor_data.cv_folds import stratified_folds\nfrom raptor_data.ready_tensors import ReadyTripletDataset\nfrom train_fusion_v2 import asset, discover_roots, load_label_table\n\nCHUNK = 16\nPER_FOLD = 150\nSEED = 20261012\nOUT = Path(\'/kaggle/working/diag\')\nRANK = int(os.environ.get(\'DIAG_RANK\', 0))\nMODES = (\'full\', \'no_fusion\', \'rad_mean\')\n\n\ndef log(kind, **data):\n    print(json.dumps(dict(event=kind, rank=RANK, utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()), **data)),\n          flush=True)\n\n\ndef kernel_dir(slug):\n    for p in (Path(\'/kaggle/input/notebooks/zafarnematov\')/slug, Path(\'/kaggle/input\')/slug):\n        if p.is_dir():\n            return p\n    raise FileNotFoundError(slug)\n\n\ndef checkpoints():\n    ck = {}\n    for k, slug in ((0, \'rsna-fusion-cv5-f01\'), (1, \'rsna-fusion-cv5-f01\'), (2, \'rsna-fusion-cv5-f23\'),\n                    (3, \'rsna-fusion-cv5-f23\'), (4, \'rsna-fusion-cv5-f4\')):\n        path = kernel_dir(slug)/f\'fold{k}\'/\'fusion_best.pt\'\n        assert path.is_file(), path\n        ck[f\'cv_fold{k}\'] = path\n    ck[\'run20_saved_best_raw_ep2\'] = next(kernel_dir(\'rsna-fusion-ddp-train\').rglob(\'fusion_best.pt\'))\n    ck[\'run20_ema_epoch20\'] = next(kernel_dir(\'rsna-fusion-ddp-resume\').rglob(\'fusion_ema_epoch20.pt\'))\n    return ck\n\n\ndef build_parent():\n    return build_raptor_v5_radimagenet(asset(\'raptor-knee-maxspan\', \'raptor_ft_coatnet_v5_full_swa.pt\'),\n                                       asset(\'resnet-50-radimagenet-marwan\', \'ResNet50.pt\'),\n                                       FusionConfig(rad_input=\'pm1_middle\'))\n\n\n# ----------------------------------------------------------------------------- weights\ndef weight_stats(parent_sd, sd):\n    groups = {\'coatnet\': (\'backbone.coatnet.\',), \'radimagenet\': (\'backbone.radimagenet.\',),\n              \'head\': (\'norm.\', \'att.\', \'clsW\', \'clsb\')}\n    is_param = lambda k: not any(s in k for s in (\'running_mean\', \'running_var\', \'num_batches_tracked\'))\n    out = {}\n    for name, prefixes in groups.items():\n        keys = [k for k in sd if k.startswith(prefixes) and is_param(k)]\n        a = torch.cat([sd[k].float().flatten() for k in keys]); b = torch.cat([parent_sd[k].float().flatten() for k in keys])\n        out[name] = dict(parameters=int(a.numel()), norm=float(a.norm()), drift_norm=float((a - b).norm()),\n                         relative_drift=float((a - b).norm()/b.norm()))\n    for name in (\'f1\', \'f2\'):\n        layers = {}\n        for idx, label in ((0, \'layernorm\'), (1, \'linear1\'), (4, \'linear2\'), (7, \'linear3_output_zero_init\')):\n            w = sd[f\'backbone.{name}.mlp.{idx}.weight\'].float(); bias = sd[f\'backbone.{name}.mlp.{idx}.bias\'].float()\n            layers[label] = dict(weight_norm=float(w.norm()), weight_rms=float(w.pow(2).mean().sqrt()),\n                                 bias_norm=float(bias.norm()),\n                                 **({\'weight_mean_minus_1\': float((w - 1).mean())} if idx == 0 else {}))\n        out[name] = layers\n    buffers = [k for k in sd if not is_param(k) and \'num_batches\' not in k]\n    out[\'batchnorm_buffers_max_abs_change\'] = max(float((sd[k].float() - parent_sd[k].float()).abs().max())\n                                                  for k in buffers)\n    return out\n\n\n# ----------------------------------------------------------------------------- activations\ndef padded_chunks(images):\n    n = len(images)\n    padded = -(-n//CHUNK)*CHUNK\n    x = images if padded == n else torch.cat([images, images[-1:].expand(padded-n, *images.shape[1:])])\n    for start in range(0, padded, CHUNK):\n        yield x[start:start+CHUNK], max(0, min(CHUNK, n-start))\n\n\n@torch.inference_mode()\ndef rad_means(model, dataset, device):\n    bb = model.backbone\n    sx = sy = None; count = 0\n    for i in range(len(dataset)):\n        for chunk, valid in padded_chunks(dataset[i][\'images\']):\n            if not valid:\n                continue\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                x, y = bb.radimagenet(rad_input_transform(chunk[:valid].to(device).float(), bb.rad_input))\n            x, y = x.float().sum(0), y.float().sum(0)\n            sx = x if sx is None else sx + x; sy = y if sy is None else sy + y; count += valid\n    return (sx/count)[None], (sy/count)[None]\n\n\ndef norm_rows(t):\n    return t.flatten(1).float().norm(dim=1)\n\n\ndef cos_rows(a, b):\n    return torch.nn.functional.cosine_similarity(a.flatten(1).float(), b.flatten(1).float(), dim=1)\n\n\n@torch.inference_mode()\ndef run_study(model, images, device, means, modes):\n    """Per-mode study probabilities + per-window statistics (full mode only)."""\n    bb = model.backbone\n    feats = {m: [] for m in modes}\n    stats = []\n    for chunk, valid in padded_chunks(images):\n        c = chunk.to(device, non_blocking=True).float()\n        with torch.autocast(\'cuda\', dtype=torch.float16):\n            x, y = bb.radimagenet(rad_input_transform(c, bb.rad_input))\n            z = bb.coatnet.stages[1](bb._stem_s1(c))\n            tail = lambda zz: bb.coatnet.stages[3](bb.coatnet.stages[2](zz))\n            pool = lambda vv: bb.coatnet.forward_head(bb.coatnet.norm(vv))\n            r1 = bb.f1(x, z)\n            v = tail(z + r1)\n            r2 = bb.f2(v, y)\n            f_full = pool(v + r2)\n            feats[\'full\'].append(f_full.float())\n            if \'no_fusion\' in modes:\n                f_nf = pool(tail(z))\n                feats[\'no_fusion\'].append(f_nf.float())\n            if \'rad_mean\' in modes:\n                xm, ym = means\n                vm = tail(z + bb.f1(xm.to(x.dtype).expand(len(c), -1, -1, -1), z))\n                f_rm = pool(vm + bb.f2(vm, ym.to(y.dtype).expand(len(c), -1, -1, -1)))\n                feats[\'rad_mean\'].append(f_rm.float())\n        if valid:\n            s = slice(0, valid)\n            row = dict(s2_residual_ratio=(norm_rows(r1[s])/norm_rows(z[s])), s2_cos=cos_rows(r1[s], z[s]),\n                       s4_residual_ratio=(norm_rows(r2[s])/norm_rows(v[s])), s4_cos=cos_rows(r2[s], v[s]))\n            if \'no_fusion\' in modes:\n                row[\'feat_change_no_fusion\'] = norm_rows(f_full[s] - f_nf[s])/norm_rows(f_full[s])\n            if \'rad_mean\' in modes:\n                row[\'feat_change_rad_mean\'] = norm_rows(f_full[s] - f_rm[s])/norm_rows(f_full[s])\n            stats.append({k: t.cpu().numpy() for k, t in row.items()})\n    n = len(images)\n    probs = {}\n    with torch.autocast(\'cuda\', dtype=torch.float16):\n        for m in modes:\n            probs[m] = torch.sigmoid(model.head(torch.cat(feats[m], 0)[:n][None]).float())[0].cpu().numpy()\n    window_stats = {k: np.concatenate([s[k] for s in stats]) for k in stats[0]}\n    return probs, window_stats\n\n\n@torch.inference_mode()\ndef run_parent(model, images, device):\n    feats = []\n    for chunk, _ in padded_chunks(images):\n        with torch.autocast(\'cuda\', dtype=torch.float16):\n            feats.append(model.backbone(chunk.to(device, non_blocking=True).float()).float())\n    with torch.autocast(\'cuda\', dtype=torch.float16):\n        return torch.sigmoid(model.head(torch.cat(feats, 0)[:len(images)][None]).float())[0].cpu().numpy()\n\n\n# ----------------------------------------------------------------------------- jobs\nclass Subset:\n    def __init__(self, dataset, indices):\n        self.dataset, self.indices = dataset, list(indices)\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, i):\n        return self.dataset[self.indices[i]]\n\n\ndef evaluate(name, model, datasets, device, modes):\n    for set_name, dataset in datasets.items():\n        started = time.time()\n        means = rad_means(model, dataset, device) if \'rad_mean\' in modes else None\n        rows, window = [], {}\n        for i in range(len(dataset)):\n            item = dataset[i]\n            probs, ws = run_study(model, item[\'images\'], device, means, modes)\n            for m, p in probs.items():\n                rows.append([item[\'study_uid\'], m, *p.tolist()])\n            for k, v in ws.items():\n                window.setdefault(k, []).append(v)\n                window.setdefault(f\'{k}__study_mean\', []).append(float(np.mean(v)))\n            window.setdefault(\'study_uid\', []).append(item[\'study_uid\'])\n        pd.DataFrame(rows, columns=[\'StudyInstanceUID\', \'mode\', *LABELS]).to_csv(\n            OUT/f\'preds_{name}_{set_name}.csv\', index=False)\n        per_window = {k: np.concatenate(v) for k, v in window.items() if not k.endswith(\'__study_mean\')\n                      and k != \'study_uid\'}\n        np.savez_compressed(OUT/f\'windows_{name}_{set_name}.npz\', **per_window)\n        pd.DataFrame({k: v for k, v in window.items() if k.endswith(\'__study_mean\') or k == \'study_uid\'}).to_csv(\n            OUT/f\'study_stats_{name}_{set_name}.csv\', index=False)\n        log(\'evaluated\', checkpoint=name, set=set_name, studies=len(dataset),\n            windows=int(len(per_window[\'s2_residual_ratio\'])), minutes=round((time.time()-started)/60, 2),\n            median={k: float(np.median(v)) for k, v in per_window.items()})\n\n\ndef main():\n    OUT.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\', 0)\n    torch.backends.cudnn.benchmark = True\n    roots = discover_roots()\n    gold = ReadyTripletDataset(roots, split=\'held_out_gold\', k=None, windows=\'pure\')\n    train = ReadyTripletDataset(roots, split=\'train\', k=None, windows=\'pure\')\n    ids = [p.stem for p in train.files]\n    binary = load_label_table(asset(\'jev-knee-labels\', \'HYBRID_labels_binary.csv\'), ids, name=\'HYBRID binary\')\n    fold_of = stratified_folds(ids, np.stack([binary[u] for u in ids]), 5, 20261010)\n    saved = pd.read_csv(kernel_dir(\'rsna-fusion-cv5-f01\')/\'fold0\'/\'cv_folds.csv\', dtype={\'StudyInstanceUID\': str})\n    assert dict(zip(saved.StudyInstanceUID, saved.fold)) == fold_of, \'fold assignment differs from training\'\n    rng = np.random.default_rng(SEED)\n    fold_sets = {}\n    for k in range(5):\n        members = [i for i, u in enumerate(ids) if fold_of[u] == k]\n        fold_sets[k] = Subset(train, sorted(rng.choice(members, PER_FOLD, replace=False).tolist()))\n    if RANK == 0:\n        pd.DataFrame([(train.files[i].stem, k) for k, s in fold_sets.items() for i in s.indices],\n                     columns=[\'StudyInstanceUID\', \'fold\']).to_csv(OUT/\'fold_subsets.csv\', index=False)\n    ck = checkpoints()\n    jobs = {0: [\'cv_fold0\', \'cv_fold1\', \'run20_saved_best_raw_ep2\', \'run20_ema_epoch20\'],\n            1: [\'cv_fold2\', \'cv_fold3\', \'cv_fold4\']}[RANK]\n    parent_folds = {0: [0, 1], 1: [2, 3, 4]}[RANK]\n    log(\'start\', jobs=jobs, parent_folds=parent_folds, gold=len(gold), per_fold=PER_FOLD)\n\n    parent = build_parent()\n    parent_sd = {k: v.detach().clone() for k, v in parent.state_dict().items()}\n    parent = parent.to(device).eval()\n    sets = {f\'fold{k}\': fold_sets[k] for k in parent_folds}\n    if RANK == 0:\n        sets = {\'gold74\': gold, **sets}\n    for set_name, dataset in sets.items():\n        rows = []\n        for i in range(len(dataset)):\n            item = dataset[i]\n            rows.append([item[\'study_uid\'], \'parent\', *run_parent(parent, item[\'images\'], device).tolist()])\n        pd.DataFrame(rows, columns=[\'StudyInstanceUID\', \'mode\', *LABELS]).to_csv(\n            OUT/f\'preds_parent_{set_name}.csv\', index=False)\n        log(\'parent_evaluated\', set=set_name, studies=len(dataset))\n    del parent\n    torch.cuda.empty_cache()\n\n    weights = {}\n    for name in jobs:\n        state = torch.load(ck[name], map_location=\'cpu\', weights_only=True)\n        meta = dict(checkpoint=str(ck[name]), selection=state.get(\'selection\'),\n                    fold=state.get(\'training_config\', {}).get(\'fold\'),\n                    rad_input=state.get(\'fusion_config\', {}).get(\'rad_input\', \'shared (absent in checkpoint)\'))\n        model = load_fused_checkpoint(state)\n        weights[name] = dict(**meta, **weight_stats(parent_sd, model.state_dict()))\n        del state\n        model = model.to(device).eval()\n        sets = {\'gold74\': gold}\n        if name.startswith(\'cv_fold\'):\n            sets[f\'fold{name[-1]}\'] = fold_sets[int(name[-1])]\n        evaluate(name, model, sets, device, MODES)\n        del model\n        torch.cuda.empty_cache()\n    (OUT/f\'weights_rank{RANK}.json\').write_text(json.dumps(weights, indent=2))\n    log(\'done\')\n\n\nif __name__ == \'__main__\':\n    main()\n'}
src = Path('/kaggle/working/fusion_src')
for name, text in SOURCES.items():
    path = src/name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
print('sources written:', sorted(SOURCES))


In [ ]:
import os, subprocess, sys, time
procs = []
for rank in range(2):
    env = dict(os.environ, PYTHONUNBUFFERED='1', OMP_NUM_THREADS='2', PYTHONPATH=str(src),
               CUDA_VISIBLE_DEVICES=str(rank), DIAG_RANK=str(rank))
    log = open(f'/kaggle/working/diag_rank{rank}.log', 'w')
    procs.append((subprocess.Popen([sys.executable, str(src/'fusion_diagnostics.py')], stdout=log,
                                   stderr=subprocess.STDOUT, env=env, cwd='/kaggle/working'), log))
while any(p.poll() is None for p, _ in procs):
    time.sleep(60)
    for rank in range(2):
        lines = open(f'/kaggle/working/diag_rank{rank}.log').read().splitlines()
        print(rank, lines[-1][:300] if lines else '', flush=True)
for rank, (p, log) in enumerate(procs):
    log.close()
    print(f'--- rank {rank} exit {p.returncode}')
    print(open(f'/kaggle/working/diag_rank{rank}.log').read()[-4000:])
if any(p.returncode for p, _ in procs):
    raise SystemExit('diagnostics failed')
